# Use LLM to enrich parsed templates with corpus‑level and sample‑level semantic descriptions and reasoning traces

This idea is taken from the paper ["Log Anomaly Detection with Large Language Models via Knowledge-Enriched Fusion"](https://arxiv.org/html/2512.11997v1)


In [8]:
import os, sys
import json
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))  # project root
load_dotenv()  # load environment variables from .env file



True

In [9]:
DATASET = "bgl"
RUNTAG = '20260921_2108'

In [10]:
from pathlib import Path

PROCESSED_DIR = Path(f"../data/processed/{DATASET.lower()}/{RUNTAG}")

# Auto-pick the newest *_{DATASET.lower()}_templates.json produced by the Parser notebook.
# Override to pin a specific run:  templates_file = PROCESSED_DIR / f"{RUNTAG}_{DATASET.lower()}_templates.json"
candidates = sorted(PROCESSED_DIR.glob(f"*_{DATASET.lower()}_templates.json"))
if not candidates:
    raise FileNotFoundError(f"No *_{DATASET.lower()}_templates.json found in {PROCESSED_DIR}")

templates_file = candidates[-1]          # lexicographic sort → newest YYYYMMDD_HHMM prefix last
# RUN_TAG = templates_file.stem.replace(f"_{DATASET.lower()}_templates", "")

print(f"Using run  : {RUNTAG}")
print(f"Input file : {templates_file}")

with open(templates_file, "r") as f:
    data = json.load(f)

templates = [entry["template"] for entry in data]

print(f"\nLoaded {len(templates)} templates")
print("First 3:")
for t in templates[:3]:
    print(" ", t)


Using run  : 20260921_2108
Input file : ../data/processed/bgl/20260921_2108/1_bgl_templates.json

Loaded 180 templates
First 3:
  RAS KERNEL INFO instruction cache parity error corrected
  RAS LINKCARD INFO MidplaneSwitchController performing bit sparing on <*> bit <*>
  RAS KERNEL <*> <*> <*>


In [11]:
from modules.enricher import Enricher, TemplateContext

# MISTRAL_LARGE_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_MISTRAL_LARGE")
# MISTRAL_SMALL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_MISTRAL_SMALL")

# DEEPSEEK_V4_FLASH_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_FLASH")
DEEPSEEK_V4_PRO_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO")

# enricher_deepseek_flash = Enricher(DEEPSEEK_V4_FLASH_DEPLOYMENT)
enricher_deepseek_pro = Enricher(DEEPSEEK_V4_PRO_DEPLOYMENT)

DATASET_CONTEXTS = {
    "hdfs": (
        "HDFS-v1 labels apply to complete traces grouped by block ID, not to an "
        "individual log event or template."
    ),
    "bgl": (
        "BGL source log messages carry event-level labels. When transformed into "
        "time windows, a window is anomalous when it contains an anomalous event."
    ),
}

def build_template_context(entry):
    return TemplateContext.from_template_record(
        entry,
        candidate_relations=entry.get("candidate_relations", []),
        retrieved_docs=entry.get("retrieved_docs", []),
        dataset_context=DATASET_CONTEXTS.get(DATASET.lower()),
    )

## Enrich with large model

In [12]:
enriched_with_large = []

for entry in data:
    enriched = enricher_deepseek_pro.enrich_template(build_template_context(entry))
    enriched_with_large.append(enriched)

## Enrich with small model

# Persist enrichments into files

In [13]:
for i, entry in enumerate(data):
    if i < len(enriched_with_large):
        entry["enriched_large"] = enriched_with_large[i].model_dump(mode="json")
    # if i < len(enriched_with_small):
    #     entry["enriched_small"] = enriched_with_small[i].model_dump(mode="json")

path_to_output = PROCESSED_DIR / f"{RUNTAG}_{DATASET.lower()}_2_templates_enriched.json"
with open(path_to_output, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved {len(data)} records → {path_to_output}")

Saved 180 records → ../data/processed/bgl/20260921_2108/20260921_2108_bgl_2_templates_enriched.json
